# SNI v3.1 - Jiao Swin-HSSAM group-primary fail-fast

Tahap pertama hanya melatih Swin-T biasa (`S3J0`) dan Swin-HSSAM lengkap (`S3J1`) pada seed 42. Checkpoint dipilih dengan Macro-F1 foto-sumber x kelas. Test tidak dibuka.

In [ ]:
SEEDS = [42]
REPO_REF = 'agent/sni-instance-crops'
DRIVE_DATA_FOLDER = 'coffee-sni-instance-crop-v1'
V3_FOLDER = 'classification-v3-source-balanced'
DRIVE_RESULT_FOLDER = 'sni-v3-jiao-group-primary-v1'
HF_REPO = 'ediprin/coffee-backbone-checkpoints'
HF_NAMESPACE = 'sni-v3-jiao-group-primary-v1'
HF_SYNC_EVERY = 1

In [ ]:
# 1/5 - Setup repository, Drive, GPU, dan checkpoint lintas akun
import csv, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
assert SEEDS == [42], 'Screening pertama hanya seed 42.'
repo = Path('/content/coffee-bean-classification')
repo_url = 'https://github.com/ediprin/coffee-bean-classification.git'

def run(command, cwd=None):
    command = [str(item) for item in command]
    print('\n$', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

if (repo / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_REF], cwd=repo)
    run(['git', 'checkout', REPO_REF], cwd=repo)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=repo)
else:
    if repo.exists(): shutil.rmtree(repo)
    run(['git', 'clone', '--branch', REPO_REF, repo_url, repo])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo])

from huggingface_hub import HfApi, login
hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Tambahkan secret HF_TOKEN dengan izin write.'
login(token=hf_token, add_to_git_credential=False)
print('HF USER:', HfApi().whoami(token=hf_token)['name'])

import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2/5 - Pulihkan crop ke disk lokal dan gunakan manifest v3 dari Drive
data_root = Path('/content/sni-instance-crops')
drive_data = Path('/content/drive/MyDrive') / DRIVE_DATA_FOLDER
manifest_root = drive_data / V3_FOLDER
if not (data_root / 'audit.json').is_file():
    shards = sorted((drive_data / 'shards').glob('crop_shard_*.tar'))
    assert (drive_data / 'complete.json').is_file() and shards, 'Backup crop belum lengkap.'
    data_root.mkdir(parents=True, exist_ok=True)
    for index, shard in enumerate(shards, 1):
        with tarfile.open(shard, 'r') as archive:
            archive.extractall(data_root, filter='data')
        if index % 5 == 0 or index == len(shards):
            print(f'RESTORE {index}/{len(shards)}', flush=True)
    for name in ('audit.json', 'manifest.csv'):
        shutil.copy2(drive_data / name, data_root / name)
assert (manifest_root / 'audit.json').is_file(), 'Manifest v3 tidak ditemukan.'
audit = json.loads((manifest_root / 'audit.json').read_text())
assert audit['status'] == 'complete' and audit['test_locked'] is True
output_root = Path('/content/drive/MyDrive') / DRIVE_RESULT_FOLDER
output_root.mkdir(parents=True, exist_ok=True)
print('IMAGES  :', data_root)
print('MANIFEST:', manifest_root)
print('OUTPUT  :', output_root)
print('TEST LOCKED:', audit['test_locked'])

In [ ]:
# 3/5 - Helper dengan heartbeat 60 detik
def load_decision(stage):
    path = output_root / f'decisions/{stage}.json'
    assert path.is_file(), f'Report belum ada: {path}'
    result = json.loads(path.read_text())
    print('\n=== PUTUSAN ===')
    print('COMPARISON:', result['comparison'])
    print('DECISION:', result['decision']['decision'])
    print('CRITERIA:', result['decision']['criteria'])
    print('TEST DIBUKA:', result['test_opened'])
    return result

def run_stage(stage, codes):
    command = [
        sys.executable, '-u', '-m',
        'bilinear_lmmd.experiments.run_sni_v3_jiao_screening',
        '--data-root', str(data_root),
        '--manifest-root', str(manifest_root),
        '--output-root', str(output_root),
        '--seeds', *map(str, SEEDS),
        '--stage', stage,
        '--evaluation-split', 'val',
        '--artifact-repo', HF_REPO,
        '--artifact-namespace', HF_NAMESPACE,
        '--artifact-sync-every', str(HF_SYNC_EVERY),
    ]
    process = subprocess.Popen(command, cwd=repo)
    started = time.monotonic()
    while process.poll() is None:
        rows = []
        for code in codes:
            history = output_root / f'outputs/{code}_seed42/history.json'
            if history.is_file():
                try: rows.append(f'{code}={len(json.loads(history.read_text()))}/50')
                except Exception: rows.append(f'{code}=saving')
        print(f'[{stage.upper()} {(time.monotonic()-started)/60:.1f} menit]', ', '.join(rows) if rows else 'inisialisasi', flush=True)
        time.sleep(60)
    assert process.wait() == 0, 'Screening gagal; baca traceback.'
    return load_decision(stage)

In [ ]:
# 4/5 - Fail-fast mekanisme Jiao: dua model saja
mechanism = run_stage('mechanism', ('S3J0', 'S3J1'))
if mechanism['decision']['decision'] != 'PASS':
    print('STOP. Jangan jalankan benchmark, seed tambahan, atau test.')

In [ ]:
# 5/5 - Opsional: EfficientNet benchmark, hanya jika mechanism PASS
mechanism = load_decision('mechanism')
assert mechanism['decision']['decision'] == 'PASS', 'STOP: mechanism FAIL.'
benchmark = run_stage('benchmark', ('S3B0',))